# Monitors under log tampering — Colab runner
Upload `swarm-log-tamper.zip` with the folder icon on the left, then run the cells in order. Add your Adaption key as a Colab secret named `ADAPTION_API_KEY` (key icon in the sidebar).

In [ ]:
!unzip -oq swarm-log-tamper.zip && cd swarm-log-tamper && pip install -q -r requirements.txt
%cd swarm-log-tamper

In [ ]:
import os
from google.colab import userdata
os.environ['ADAPTION_API_KEY'] = userdata.get('ADAPTION_API_KEY')

## 1. Offline smoke test (free; numbers are fake)

In [ ]:
!python run_pipeline.py all --backend mock --run runs/mock --n-attack 5 --n-benign 5 --n-boot 300 | tail -25

## 2. Check which models your key can call

In [ ]:
!python run_pipeline.py models

## 3. Pilot: one episode of each kind per scenario. Read the example before scaling up.

In [ ]:
!python run_pipeline.py generate --run runs/pilot --n-attack 1 --n-benign 1
!python run_pipeline.py tamper --run runs/pilot
!python run_pipeline.py monitor --run runs/pilot
!python run_pipeline.py show --run runs/pilot --episode payments_release-attack-000 --condition rewrite_solo

## 4. Main run (30 attack + 30 benign episodes). Cached, so re-running a cell resumes.

In [ ]:
!python run_pipeline.py all --run runs/main --n-attack 10 --n-benign 10 --workers 12

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('runs/main/results.md').read()))
display(Image('runs/main/roc.png'))

## 5. Download results

In [ ]:
!zip -qr results.zip runs/main/results.md runs/main/results.json runs/main/roc.png runs/main/scores.jsonl runs/main/examples
from google.colab import files
files.download('results.zip')